In [1]:
import glob
import os
import pandas as pd
import re
import sys

if '../' not in sys.path:
    sys.path.append('../')

from emp.dataset import extract_bl_shelfmark, gen_prompt

In [2]:
full_aac_df = pd.read_csv("../data/external/full_aac_by_date.csv", encoding="utf8", index_col=0)
boll_emp_df = pd.read_csv("../data/external/boll_emp_by_date.csv", encoding="utf8", skipfooter=3, engine='python', usecols=[0, 1, 2, 3]).drop_duplicates(subset=["shelfmark", "title_date"])
boll_emp_df["shelfmark"] = boll_emp_df["shelfmark"].str.strip()
boll_emp_df.set_index("shelfmark", inplace=True)
boll_emp_df.rename(index={"14625.d. 16": "14625.d.16"}, inplace=True)
assert boll_emp_df.index[-1] == "14625.e.5"

In [3]:
# modifications to boll_emp_df
# 22/06/26 have double checked all these corrections are applied to the correct cells
boll_emp_df.loc["14620.k.1", "title_date"] = "Anbiya 1897"
boll_emp_df.loc["14620.g.11", "title_date"] = "Bidayat al-Salikin 1906"
boll_emp_df.loc["14620.d.20(1)", "title_date"] = "Bustan Arifin 1820-1822"
boll_emp_df.loc["14625.d.14", "title_date"] = "Dandan Setia 1894"
boll_emp_df.loc["14625.c.47", "title_date"] = "Dermah Tasiah 1906"
boll_emp_df.loc["14626.c.14(4)", "title_date"] = "Haris Fadhillah 1900"
boll_emp_df.loc["14620.ff.1", "title_date"] = "Hidayat al-Awamm 1903"
boll_emp_df.loc["14620.g.17", "title_date"] = "Hilyat al-Anam 1893"
boll_emp_df.loc["Jav.76", "title_date"] = "Husn al-Akhlak 1900"
boll_emp_df.loc["14620.b.14(8)", "title_date"] = "Ilmu Kepandaian 1843"
boll_emp_df.loc["14629.e.2", "title_date"] = "Ilmu Kepandaian a 1839"
boll_emp_df.loc["14519.d.44(1)", "title_date"] = "Maulud 1871.a"
boll_emp_df.loc["14519.d.9(3)", "title_date"] = "Maulud 1871.b"
boll_emp_df.loc["14620.g.7", "title_date"] = "Miftah al-Bayan 1890"
boll_emp_df.loc["14620.d.17(4)", "title_date"] = "Pelajaran Bahasa Melayu (No.1) b 1838/9"
boll_emp_df.loc["14620.f.5", "title_date"] = "Tahsil al-Ujur 1893"
boll_emp_df.loc["14620.g.5", "title_date"] = "Targhib al-Nas 1873"
boll_emp_df.loc["14626.a.37", "title_date"] = "Umm al-Burhan a"

In [4]:
title_loc_path = "../data/interim/title_loc_df.csv"
title_loc_df = pd.read_csv(title_loc_path, encoding="utf-8-sig", index_col=0)

In [5]:
title_loc_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 924 entries, Abbas to Zubaidah
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   short_title_desc  924 non-null    object 
 1   entry_start       924 non-null    int64  
 2   min_line          924 non-null    int64  
 3   max_line          924 non-null    int64  
 4   nearest_line      924 non-null    object 
 5   similarity        924 non-null    float64
 6   nearest_line_idx  924 non-null    int64  
 7   entry_end         924 non-null    int64  
 8   correct_title     924 non-null    object 
 9   entry_text        912 non-null    object 
dtypes: float64(1), int64(5), object(4)
memory usage: 79.4+ KB


In [7]:
def extract_title(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return " ".join(s.split()[:-2])
    elif date_re.search(s.split()[-1]):
        return " ".join(s.split()[:-1]) 
    elif s.split()[-1] in ["a", "b", "c"]:
        return " ".join(s.split()[:-1])
    else:
        return s

In [8]:
def extract_edition(s):
    date_re = re.compile(r"\d{4,4}")
    if date_re.search(s.split()[-1]) and s.split()[-2] in ["a", "b", "c"]:
        return s.split()[-2]
    elif date_re.search(s.split()[-1]):
        return s.split()[-1] 
    elif s.split()[-1] in ["a", "b", "c"]:
        return s.split()[-1]
    else:
        return None

In [9]:
boll_emp_df["title"] = boll_emp_df["title_date"].apply(lambda x: extract_title(x))
boll_emp_df["edition"] = boll_emp_df["title_date"].apply(lambda x: extract_edition(x))

In [10]:
assert not [x for x in boll_emp_df.set_index(["title"]).index.difference(title_loc_df.index)]
assert not boll_emp_df["edition"].hasnans
assert title_loc_df.index.is_unique

In [11]:
title_loc_df["entry_start"].value_counts().head(8)

entry_start
14768    2
36769    2
40345    2
41561    2
43355    2
44525    2
44228    2
47551    2
Name: count, dtype: int64

In [12]:
boll_entry_df = title_loc_df.merge(boll_emp_df, how="inner", left_index=True, right_on="title").reset_index()
assert boll_entry_df.drop_duplicates(subset="title")["entry_start"].is_monotonic_increasing
boll_entry_df.set_index("title", inplace=True)
assert not boll_entry_df["entry_text"].hasnans

In [13]:
boll_entry_df

,shelfmark,short_title_desc,entry_start,min_line,max_line,nearest_line,similarity,nearest_line_idx,entry_end,correct_title,entry_text,title_date,year,pages,edition
title,,,,,,,,,,,,,,,
Abdau,14516.b.59,Abdau,42,0,2000,Abdau,100.000000,42,90,Abdau,"Abdau\n1896\nKitab Abdau :tp,col; Nazam Melayu...",Abdau 1896,1896,21,1896
Abdullah dan Sabat,14620.b.13(6),Abdullah dan Sa bat,572,91,2091,Abdullah dan Sabat,97.297297,572,676,Abdullah dan Sa bat,Abdullah dan Sabat\n1829\nCeritera Hikayat Abd...,Abdullah dan Sabat 1837.b,1837,24,1837.b
Abdul Muluk,14625.e.15,Abdul Muluk,677,91,2091,AbdulMuluk,95.238095,677,1090,Abdul Muluk,AbdulMuluk\na 1860.\n[? 1860s]\nContents:\nthe...,Abdul Muluk 1874,1874,202,1874
Abu Syahmah,14620.g.19(8),Abu Syahmah,1263,1177,3177,Abu Syahmah,100.000000,1263,1452,Abu Syahmah,Abu Syahmah\na :1:J87J\n[1st edition]\nHikayat...,Abu Syahmah 1901,1901,36,1901
Agama Kitab,14620.b.12(5),Agama Kitab,1787,1770,3770,AgamaKitab,95.238095,1787,1886,Agama Kitab,AgamaKitab\npublisher: Mission Press\nSingapor...,Agama Kitab a c.1835,1835,8,a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Yusuf,14620.b.14(5),Yusuf,50294,50029,52029,Yusuf,100.000000,50294,50480,Yusuf,Yusuf\n[? Singapore]\n[ca. 1841]\n22pp. (1-22)...,Yusuf a c.1841,1841,22,a
Zabur,14620.b.13(2),Zabur,50481,50294,52294,Zabur,100.000000,50481,50556,Zabur,Zabur\n1824\nPuji-Pujian iaitu Segala Zabur Da...,Zabur 1824,1824,43,1824
Zhongjieyi,14625.a.5,Zbongjieyi,50691,50481,52481,Zbongjieyi,90.000000,50691,50812,Zhongjieyi,"Zbongjieyi\n1889\n""Charita dahulu kala yang be...",Zhongjieyi 1889,1889,145,1889


In [14]:
boll_entry_df["extracted_shelfmark"] = boll_entry_df["entry_text"].apply(lambda x: extract_bl_shelfmark(x)).str.strip().str.replace("v. ", "v.")

In [15]:
absent_raw_sm_df = boll_entry_df[~(boll_entry_df.apply(lambda x: x["shelfmark"] in x["entry_text"], axis=1))]
absent_sm_df = absent_raw_sm_df[absent_raw_sm_df["shelfmark"] != absent_raw_sm_df["extracted_shelfmark"]]

In [16]:
# checking that extract_bl_shelfmark will find the correct shelfmark
# correct shelfmark -> shelfmark in entry -> entry sm will be corrected by extract_bl_shelfmark

# 14620.k.1 -> 14620.k.l -> yes
# Jav.73 -> Jav. 73 -> yes

# 14626.c.10 -> 14626.c.1O -> yes

# 14653.d.1 -> 14653.d.l -> no  # but only because there are multiple shelfmarks for this edition, will probably have to pick this up in post

# 14626.a.10 -> 14626.a.1O -> yes

# 14625.e.10 -> 14625.e.1O -> yes

# Jav.70 -> Jav. 70 -> yes

# 14626.d.11(2) -> 14626.d.ll(2) -> yes

# 14620.c.10 -> 14620.c.1O -> yes

# ORB.30/612 -> ORB. 30/612 -> yes

# 14626.d.11(10) -> 14626.d.11(1O) -> yes

# 14653.b.1 -> 14653.b.l -> yes

# 14620.d.19(14) -> 14620.d.19(l4) -> yes

# 14625.a.1 -> 14625.a.l -> yes

# 14626.d.11(6) -> 14626.d.ll(6) -> yes (unclear why not picked up already)

# 14626.d.10 -> 14626.d.1O -> yes
# ORB.30/452 -> ORB. 30/452 -> yes

# fixed by modifying entry in extract_clean_entries

# 14620.a.19(10) -> 14620.aI9(10) -> no  # sm missing a '.', will need to add in
# 14620.b.18(10) -> 14620.b.18{l0) -> no  # will need to modify shelfmark
# 14623.c.4 -> 14623.cA -> no  # need to modify shelfmark
# 14626.d.11(8) -> 14626.d.l1 (8) -> no  # may need to modify sm to remove space
# 14625.a.9 -> 14625.a9 -> no  # may need to add missing '.'
# 14620.g.20(0) -> 14620.g.20(-) -> no  # may need to modify sm to swap -/0
# 14626.e.4 -> 14626.eA -> no  # will need to add '.' to sm and swap A/4
# 14633.a.38 -> 1463303.38 -> no  # will need to fix sm

# 14620.g.11 -> -> no  # may need to modify entry extent
# 14620.d.14 -> -> no  # may need to modify entry extend
# 14625.e.13 -> -> no  # need to modify entry extent
# 14629.c.31 -> -> no  # no, may need to modify entry
# 14620.g.19(9) ->  -> no  # may need to modify entry extent
# 14620.d.11 ->  -> no  # may need to modify entry

In [ ]:
# with open("../data/interim/sm_check.txt", "w") as f:
#     [f.write(r[1]["shelfmark"] + "\n" + r[1]["extracted_shelfmark"] + "\n\n" + r[1]["entry_text"] + "\n\n") for r in absent_sm_df.iterrows()]

In [ ]:
assert len(absent_sm_df) == 17  # I've accounted for these 17

In [ ]:
boll_entry_df[~(boll_entry_df.apply(lambda x: x["edition"] in x["entry_text"], axis=1))]

In [ ]:
boll_entry_df.loc["Agama Kitab"]

In [ ]:
boll_entry_df.loc["Agama Kitab", "entry_text"]

### Export prepared entries to interim

In [ ]:
boll_entry_df.drop_duplicates(subset="title").to_csv("../data/interim/title_dedup_boll_emp.csv", encoding="utf-8-sig", index=False)

In [ ]:
pd.read_csv("../data/interim/title_dedup_boll_emp.csv", encoding="utf-8-sig").set_index("title")